# Marine 48h Forecast — Hybrid v2: iTransformer + Distribution-Correct XGBoost

The first hybrid attempt (`Marine_Forecast_RealEMS_Hybrid_iTransformer_DeepAR.ipynb`)
swapped DeepAR in for the 6 parameters where a single iTransformer failed catastrophically
(skill −100% to −410%). It helped — average +210 percentage points — but only 2 of 6
actually turned positive; the other 4 just landed near persistence. **That's a fair
"hasn't really helped" verdict**: DeepAR's Gaussian likelihood is still the wrong
distributional assumption for these 6 parameters' actual shapes.

## Checking the actual distributions first

| Parameter | Shape | Evidence |
|---|---|---|
| `precipitationIntensity` | Zero-inflated, skewed positive tail | 96.5% exact zeros, max 7.92 mm/h |
| `precipitationDifference` | Zero-inflated, skewed positive tail | 82.0% exact zeros, max 12.55 mm |
| `oneMinuteAvgVisibility`, `tenMinuteAvgVisibility`, `oneHourAvgVisibility`, `twentyFourHourAvgVisibility` | Ceiling-saturated, rare large drops | Mean ≈ 18,084, sitting right at max ≈ 18,800-18,846; **no zeros at all** |

DeepAR assumes every channel is approximately Gaussian — true for neither shape here.
Precipitation's zero-inflation needs a **zero-inflated / compound distribution** model;
visibility's ceiling-saturation needs a model that **physically cannot extrapolate**
past the observed range, which a Gaussian-likelihood RNN's recursive rollout can (and
did) do.

## The fix: XGBoost, with the right loss function for each shape

- **Precipitation (2 parameters)**: `objective="reg:tweedie"` — the standard fit for
  "rain or no rain, and how much" in hydrology/actuarial literature (compound
  Poisson-Gamma; Dunn 2004; `tweedie_variance_power=1.5` is the conventional middle-ground
  setting between pure Poisson (1.0) and pure Gamma (2.0)).
- **Visibility (4 parameters)**: `objective="reg:pseudohubererror"` — robust to the rare
  large drops without being dominated by them, on a tree-based model that's bounded by
  construction (a tree's leaf prediction can never exceed the range of training targets
  it was fit on, unlike an RNN's recursive rollout).

iTransformer's responsibility (the 18 "good" parameters) is unchanged from the previous
hybrid. Fully standalone — does not modify or depend on any other notebook/dashboard.

## 0. Setup

In [ ]:
import time
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cpu")
torch.set_num_threads(8)

print("PyTorch:", torch.__version__, "| XGBoost:", xgb.__version__, "| torch threads:", torch.get_num_threads())


## 1. Load data, collapse duplicates, encode circular parameters

In [ ]:
df_10min = pd.read_csv("ems_10min_resampled.csv", index_col=0, parse_dates=True)
DUPLICATES = [
    ("airTemperature", "windChillTemperature"),
    ("tideLevel", "tidePressure"),
    ("tideLevel", "waterPressure"),
    ("tideLevel", "waterLevel"),
    ("waterTemperature", "waterTemperature_WQ"),
    ("significantWaveHeight", "maxWaveHeight"),
]
df_cat = df_10min[["precipitationType"]].copy()
df_num = df_10min.drop(columns=["precipitationType"]).copy()

CIRCULAR = ["windDirection", "currentDirection", "compass"]
for c in CIRCULAR:
    rad = np.deg2rad(df_num[c])
    df_num[f"{c}_sin"] = np.sin(rad)
    df_num[f"{c}_cos"] = np.cos(rad)
df_num_full = df_num.drop(columns=CIRCULAR)

target_cols = [c for c in df_num_full.columns if c not in [d for _, d in DUPLICATES]]
n_targets = len(target_cols)
print(f"Modeled channels: {n_targets}")

PRECIP_PARAMS = ["precipitationIntensity", "precipitationDifference"]
VISIBILITY_PARAMS = ["twentyFourHourAvgVisibility", "tenMinuteAvgVisibility",
                      "oneMinuteAvgVisibility", "oneHourAvgVisibility"]
HARD_PARAMS = PRECIP_PARAMS + VISIBILITY_PARAMS
GOOD_PARAMS = [c for c in target_cols if c not in HARD_PARAMS]
print(f"iTransformer: {len(GOOD_PARAMS)} parameters | XGBoost (Tweedie/Huber): {len(HARD_PARAMS)} parameters")


## 2. Train/test split, duplicate reconstruction fit, scaling

In [ ]:
LOOKBACK, HORIZON = 288, 288   # 2 days lookback, 48h horizon @ 10-min steps

idx = df_num_full.index
df_num_full["hour_sin"] = np.sin(2 * np.pi * idx.hour / 24)
df_num_full["hour_cos"] = np.cos(2 * np.pi * idx.hour / 24)
df_num_full["dom_sin"] = np.sin(2 * np.pi * idx.day / 30)
df_num_full["dom_cos"] = np.cos(2 * np.pi * idx.day / 30)
calendar_cols = ["hour_sin", "hour_cos", "dom_sin", "dom_cos"]

feature_cols = target_cols + calendar_cols
model_data = df_num_full[feature_cols].copy()
n_features = len(feature_cols)
target_idx = [feature_cols.index(c) for c in target_cols]
good_idx = [feature_cols.index(c) for c in GOOD_PARAMS]
calendar_idx = [feature_cols.index(c) for c in calendar_cols]

train_df = model_data.iloc[:-HORIZON].copy()
test_df = model_data.iloc[-HORIZON:].copy()
mean, std = train_df.mean(), train_df.std().replace(0, 1)
train_scaled = (train_df - mean) / std
full_scaled = (model_data - mean) / std

print(f"Train: {train_df.shape[0]} rows ({train_df.shape[0]/144:.1f} days)")
print(f"Test : {test_df.shape[0]} rows ({test_df.shape[0]/144:.1f} days)")


In [ ]:
recon_coef = {}
for keep, drop in DUPLICATES:
    x, y = train_df[keep].values, df_num_full[drop].iloc[:-HORIZON].values
    slope, intercept = np.polyfit(x, y, 1)
    pred_train = slope * x + intercept
    r2 = 1 - np.sum((y - pred_train) ** 2) / np.sum((y - y.mean()) ** 2)
    recon_coef[drop] = (keep, float(slope), float(intercept), float(r2))
    print(f"  reconstruct {drop:25s} = {slope:.4f} * {keep} + {intercept:.4f}   (R^2={r2:.5f})")


## 3. Model A — iTransformer (18 "good" parameters, unchanged from Hybrid v1)

In [ ]:
def make_direct_windows(scaled_df, lookback, horizon, out_idx):
    arr = scaled_df.values.astype(np.float32)
    X, Y = [], []
    for origin in range(lookback, len(arr) - horizon):
        X.append(arr[origin - lookback:origin])
        Y.append(arr[origin:origin + horizon][:, out_idx])
    return np.array(X, dtype=np.float32), np.array(Y, dtype=np.float32)

X_direct, Y_good = make_direct_windows(train_scaled, LOOKBACK, HORIZON, good_idx)
X_t, Y_good_t = torch.from_numpy(X_direct), torch.from_numpy(Y_good)
n_val = max(1, int(0.1 * len(X_t)))
X_tr, Y_tr_good = X_t[:-n_val], Y_good_t[:-n_val]
X_val, Y_val_good = X_t[-n_val:], Y_good_t[-n_val:]
last_window = torch.from_numpy(train_scaled.values[-LOOKBACK:].astype(np.float32)).unsqueeze(0)


class ITransformer(nn.Module):
    def __init__(self, lookback, n_features, horizon, out_idx, d_model=64, n_heads=4,
                 n_layers=2, dropout=0.1):
        super().__init__()
        self.out_idx = out_idx
        self.embed = nn.Linear(lookback, d_model)
        self.var_id = nn.Parameter(torch.randn(n_features, d_model) * 0.02)
        layer = nn.TransformerEncoderLayer(d_model, n_heads, dim_feedforward=d_model * 2,
                                            dropout=dropout, batch_first=True, activation="gelu")
        self.encoder = nn.TransformerEncoder(layer, n_layers)
        self.head = nn.Linear(d_model, horizon)

    def forward(self, x):
        tok = self.embed(x.transpose(1, 2)) + self.var_id.unsqueeze(0)
        tok = self.encoder(tok)
        out = self.head(tok)
        return out.transpose(1, 2)[:, :, self.out_idx]

    def embed_tokens(self, x):
        return self.embed(x.transpose(1, 2)) + self.var_id.unsqueeze(0)


def train_model(model, X_tr, Y_tr, X_val, Y_val, epochs=150, batch_size=64, lr=1e-3,
                 patience=20, name=""):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=6)
    loss_fn = nn.MSELoss()
    best_val, best_state, wait = float("inf"), None, 0
    n = len(X_tr); t0 = time.time()
    for ep in range(epochs):
        ep_t0 = time.time()
        model.train()
        perm = torch.randperm(n)
        for i in range(0, n, batch_size):
            b = perm[i:i + batch_size]
            xb, yb = X_tr[b].to(device), Y_tr[b].to(device)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(X_val.to(device)), Y_val.to(device)).item()
        sched.step(val_loss)
        print(f"  [{name}] epoch {ep+1:3d}/{epochs}  val_loss={val_loss:.4f}  "
              f"epoch_time={time.time()-ep_t0:.1f}s  elapsed={time.time()-t0:.0f}s", flush=True)
        if val_loss < best_val - 1e-5:
            best_val, wait = val_loss, 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience: break
    if best_state is not None: model.load_state_dict(best_state)
    model.eval()
    print(f"{name:14s} best_val_loss={best_val:.4f}  epochs_run={ep+1:3d}  time={time.time()-t0:5.1f}s")
    return model

itransformer = ITransformer(LOOKBACK, n_features, HORIZON, good_idx, d_model=64, n_heads=4, n_layers=2)
itransformer = train_model(itransformer, X_tr, Y_tr_good, X_val, Y_val_good, epochs=150, patience=20,
                            name="iTransformer")

with torch.no_grad():
    good_pred_scaled = itransformer(last_window.to(device))[0].cpu().numpy()
good_preds_real = good_pred_scaled * std[GOOD_PARAMS].values + mean[GOOD_PARAMS].values
good_pred_df = pd.DataFrame(good_preds_real, columns=GOOD_PARAMS, index=test_df.index)
print("iTransformer 48h forecast complete (18 parameters).")


## 4. Model B — XGBoost, Tweedie loss (precipitation) and Pseudo-Huber loss (visibility)

Direct multi-horizon, one model per parameter, lag + lead-time + calendar features (same
recipe as the 11-model notebook's XGBoost section) — but **two different scales**:

- **Precipitation**: trained on **raw (unscaled)** values. Tweedie's variance-power
  assumption is about the response's actual non-negative scale (it's a compound
  Poisson-Gamma distribution defined for y ≥ 0) — z-scoring would introduce negative
  values that violate Tweedie's domain.
- **Visibility**: trained on **z-scored (standardized)** values. `reg:pseudohubererror`
  turned out to be numerically unstable at visibility's raw scale (~18,000) — an early
  boosting round produced one wildly miscalibrated constant leaf value that every
  subsequent round failed to correct, leaving every prediction collapsed to the same
  absurd constant (confirmed by direct inspection: all 288 test predictions identical at
  ~3.1×10¹⁰). Standardizing first keeps residuals in a numerically sane range for the
  Huber loss's linear/quadratic transition.

In [ ]:
ORIGIN_LAGS = [1, 2, 3, 6, 12, 24, 48, 72, 144, 288]

def make_direct_training_xgb(df, cols, calendar_cols, lags, horizon, origin_step=16):
    # Builds lag+lead_h+calendar features and targets, all sourced from the SAME df
    # (either raw train_df for Tweedie-appropriate non-negative targets, or train_scaled
    # for numerically-sane standardized targets).
    n, max_lag = len(df), max(lags)
    rows = {c: [] for c in cols}
    feats = []
    for origin in range(max_lag, n - horizon, origin_step):
        base = {f"{c}_lag{L}": df[c].iloc[origin - L] for c in cols for L in lags}
        for h in range(1, horizon + 1, 2):
            row = dict(base); row["lead_h"] = h
            for cc in calendar_cols:
                row[cc] = df[cc].iloc[origin + h]
            feats.append(row)
            for c in cols:
                rows[c].append(df[c].iloc[origin + h])
    return pd.DataFrame(feats), {c: np.array(rows[c]) for c in cols}

X_precip, Y_precip = make_direct_training_xgb(train_df, PRECIP_PARAMS, calendar_cols,
                                                ORIGIN_LAGS, HORIZON, origin_step=16)
X_vis, Y_vis = make_direct_training_xgb(train_scaled, VISIBILITY_PARAMS, calendar_cols,
                                          ORIGIN_LAGS, HORIZON, origin_step=16)
print(f"Precipitation (raw) training rows: {X_precip.shape[0]:,}  features: {X_precip.shape[1]}")
print(f"Visibility (scaled) training rows: {X_vis.shape[0]:,}  features: {X_vis.shape[1]}")


In [ ]:
xgb_models = {}
for c in PRECIP_PARAMS:
    m = xgb.XGBRegressor(
        n_estimators=180, max_depth=5, learning_rate=0.06, subsample=0.8, colsample_bytree=0.8,
        random_state=SEED, n_jobs=4, tree_method="hist",
        objective="reg:tweedie", tweedie_variance_power=1.5,
    )
    m.fit(X_precip, Y_precip[c])
    xgb_models[c] = m
    print(f"  trained {c:30s} objective=reg:tweedie (raw scale)")

for c in VISIBILITY_PARAMS:
    m = xgb.XGBRegressor(
        n_estimators=180, max_depth=5, learning_rate=0.06, subsample=0.8, colsample_bytree=0.8,
        random_state=SEED, n_jobs=4, tree_method="hist",
        objective="reg:pseudohubererror",
    )
    m.fit(X_vis, Y_vis[c])
    xgb_models[c] = m
    print(f"  trained {c:30s} objective=reg:pseudohubererror (standardized scale)")


In [ ]:
origin_idx = len(train_df) - 1

# Precipitation forecast (raw scale)
base_row_p = {f"{c}_lag{L}": train_df[c].iloc[origin_idx - (L - 1)] for c in PRECIP_PARAMS for L in ORIGIN_LAGS}
pred_rows_p = []
for h in range(1, HORIZON + 1):
    ts = test_df.index[h - 1]
    row = dict(base_row_p); row["lead_h"] = h
    for cc in calendar_cols:
        row[cc] = model_data.loc[ts, cc]   # raw-scale calendar, matching train_df's source
    pred_rows_p.append(row)
X_fore_p = pd.DataFrame(pred_rows_p)[list(X_precip.columns)]

# Visibility forecast (standardized scale)
base_row_v = {f"{c}_lag{L}": train_scaled[c].iloc[origin_idx - (L - 1)] for c in VISIBILITY_PARAMS for L in ORIGIN_LAGS}
pred_rows_v = []
for h in range(1, HORIZON + 1):
    ts = test_df.index[h - 1]
    row = dict(base_row_v); row["lead_h"] = h
    for cc in calendar_cols:
        row[cc] = full_scaled.loc[ts, cc]
    pred_rows_v.append(row)
X_fore_v = pd.DataFrame(pred_rows_v)[list(X_vis.columns)]

hard_pred_df = pd.DataFrame(index=test_df.index)
for c in PRECIP_PARAMS:
    hard_pred_df[c] = xgb_models[c].predict(X_fore_p)
    hard_pred_df[c] = hard_pred_df[c].clip(lower=0)   # precipitation can't be negative
for c in VISIBILITY_PARAMS:
    pred_scaled_c = xgb_models[c].predict(X_fore_v)
    hard_pred_df[c] = pred_scaled_c * std[c] + mean[c]   # back to raw visibility units

print("XGBoost (Tweedie/Huber) 48h forecast complete (6 parameters).")
print(hard_pred_df.describe().T[["mean", "min", "max"]])


## 5. Merge into the hybrid forecast, reconstruct circular params & duplicates

In [ ]:
hybrid_pred_df = pd.concat([good_pred_df, hard_pred_df], axis=1)[target_cols]

def reconstruct(pred_df_in):
    out = pred_df_in.copy()
    for ang in ["windDirection", "currentDirection", "compass"]:
        out[ang] = (np.rad2deg(np.arctan2(out[f"{ang}_sin"], out[f"{ang}_cos"])) % 360)
    return out

hybrid_final = reconstruct(hybrid_pred_df)
truth = df_num_full.iloc[-HORIZON:].copy()
for ang in ["windDirection", "currentDirection", "compass"]:
    truth[ang] = (np.rad2deg(np.arctan2(truth[f"{ang}_sin"], truth[f"{ang}_cos"])) % 360)

report_params = [c for c in target_cols if not c.endswith(("_sin", "_cos"))] + \
                ["windDirection", "currentDirection", "compass"]
CIRCULAR_PARAMS = {"windDirection", "currentDirection", "compass"}
ENGINE = {p: ("XGBoost" if p in HARD_PARAMS else "iTransformer") for p in report_params}

dup_series = {}
for keep, drop in DUPLICATES:
    _, slope, intercept, _ = recon_coef[drop]
    dup_series[drop] = slope * hybrid_final[keep].values + intercept
    ENGINE[drop] = ENGINE[keep]

print("Hybrid v2 forecast assembled: 18 from iTransformer + 6 from XGBoost (Tweedie/Huber) + 6 duplicates.")


## 6. Score against persistence, pure-iTransformer, and the DeepAR-hybrid baseline

In [ ]:
PURE_ITRANSFORMER_SKILL = {
    "currentDirection": 95.6, "windDirection": 95.5, "airTemperature": 95.4,
    "relativeHumidity": 93.6, "dewPointTemperature": 92.7, "tideLevel": 92.1,
    "waterTemperature": 92.0, "significantWaveHeight": 88.4, "windSpeed": 88.2,
    "airPressure": 87.2, "globalRadiation": 86.9, "currentSpeed": 83.9,
    "salinity": 78.7, "significantWavePeriod": 74.1, "zeroCrossingPeriod": 72.2,
    "compass": 67.6, "peakWaveEnergyPeriod": 58.0, "conductivity": 57.8,
    "twentyFourHourAvgVisibility": -100.0, "precipitationDifference": -101.9,
    "tenMinuteAvgVisibility": -154.9, "oneMinuteAvgVisibility": -190.5,
    "oneHourAvgVisibility": -291.6, "precipitationIntensity": -409.9,
}
DEEPAR_HYBRID_SKILL = {   # from Marine_Forecast_RealEMS_Hybrid_iTransformer_DeepAR.ipynb
    "tenMinuteAvgVisibility": 14.0, "twentyFourHourAvgVisibility": 3.5,
    "precipitationDifference": -0.1, "precipitationIntensity": -0.2,
    "oneHourAvgVisibility": -1.4, "oneMinuteAvgVisibility": -2.5,
}

def circ_mae(true, pred):
    return np.abs((true - pred + 180) % 360 - 180).mean()

last_obs = df_num_full.iloc[-HORIZON - 1]
for ang in ["windDirection", "currentDirection", "compass"]:
    last_obs[ang] = (np.rad2deg(np.arctan2(last_obs[f"{ang}_sin"], last_obs[f"{ang}_cos"])) % 360)

metrics = []
for p in report_params:
    yt = truth[p].values
    yp_persist = np.repeat(last_obs[p], HORIZON)
    is_circular = p in CIRCULAR_PARAMS
    mae_p = circ_mae(yt, yp_persist) if is_circular else mean_absolute_error(yt, yp_persist)
    yhat = hybrid_final[p].values
    if is_circular:
        mae, rmse = circ_mae(yt, yhat), np.nan
    else:
        mae = mean_absolute_error(yt, yhat)
        rmse = np.sqrt(mean_squared_error(yt, yhat))
    skill = (1 - mae / mae_p) * 100 if mae_p > 0 else np.nan
    pure_skill = PURE_ITRANSFORMER_SKILL.get(p, np.nan)
    deepar_skill = DEEPAR_HYBRID_SKILL.get(p, np.nan)
    metrics.append({
        "parameter": p, "engine": ENGINE[p], "Persistence_MAE": round(mae_p, 4),
        "hybrid_v2_MAE": round(mae, 4), "hybrid_v2_RMSE": round(rmse, 4) if rmse == rmse else np.nan,
        "hybrid_v2_skill_%": round(skill, 1), "pure_iTransformer_skill_%": pure_skill,
        "deepar_hybrid_skill_%": deepar_skill,
        "improvement_vs_pure_pp": round(skill - pure_skill, 1) if pure_skill == pure_skill else np.nan,
        "improvement_vs_deepar_pp": round(skill - deepar_skill, 1) if deepar_skill == deepar_skill else np.nan,
    })

metrics_df = pd.DataFrame(metrics).sort_values("hybrid_v2_skill_%", ascending=False).reset_index(drop=True)
metrics_df.insert(0, "rank", metrics_df.index + 1)
metrics_df.to_csv("metrics_hybrid_v2.csv", index=False)
metrics_df


## 7. The verdict: did the distribution-correct loss actually fix the hard 6?

In [ ]:
hard_comparison = metrics_df[metrics_df["parameter"].isin(HARD_PARAMS)][
    ["parameter", "hybrid_v2_skill_%", "pure_iTransformer_skill_%", "deepar_hybrid_skill_%",
     "improvement_vs_pure_pp", "improvement_vs_deepar_pp"]
].sort_values("hybrid_v2_skill_%", ascending=False)
print(hard_comparison.to_string(index=False))
print(f"\nMean skill, hard 6 -- pure iTransformer: {metrics_df[metrics_df['parameter'].isin(HARD_PARAMS)]['pure_iTransformer_skill_%'].mean():.1f}%")
print(f"Mean skill, hard 6 -- DeepAR hybrid:      {metrics_df[metrics_df['parameter'].isin(HARD_PARAMS)]['deepar_hybrid_skill_%'].mean():.1f}%")
print(f"Mean skill, hard 6 -- XGBoost hybrid (v2): {hard_comparison['hybrid_v2_skill_%'].mean():.1f}%")
n_positive = int((hard_comparison["hybrid_v2_skill_%"] > 0).sum())
print(f"\nParameters beating persistence: {n_positive} / 6")


## 8. Plot the hard 6, v2 vs actual

In [ ]:
hist_tail = df_10min.iloc[-HORIZON - LOOKBACK:-HORIZON]
fig, axes = plt.subplots(2, 3, figsize=(18, 8))
for ax, p in zip(axes.ravel(), HARD_PARAMS):
    ax.plot(hist_tail.index, hist_tail[p], color="0.6", lw=1, label="history")
    ax.plot(truth.index, truth[p], color="black", lw=2.2, label="actual")
    ax.plot(truth.index, hybrid_final[p], color="#17becf", lw=1.5, ls="--", label="XGBoost (v2)")
    ax.axvline(truth.index[0], color="green", lw=1, alpha=0.5)
    row = metrics_df[metrics_df["parameter"] == p].iloc[0]
    ax.set_title(f"{p}\nskill: pure={row['pure_iTransformer_skill_%']:.0f}% deepar={row['deepar_hybrid_skill_%']:.0f}% v2={row['hybrid_v2_skill_%']:.0f}%",
                 fontsize=9)
    ax.grid(alpha=0.3); ax.tick_params(axis="x", rotation=30, labelsize=8)
axes.ravel()[0].legend(fontsize=8, loc="upper left")
fig.suptitle("Hard 6 Parameters — Hybrid v2 (XGBoost Tweedie/Huber) vs Actual", fontsize=13, y=1.02)
fig.tight_layout()
fig.savefig("forecast_plots_hybrid_v2.png", dpi=110, bbox_inches="tight")
plt.show()


## 9. Attention map (iTransformer's 18 parameters, unchanged from Hybrid v1)

In [ ]:
itransformer.eval()
with torch.no_grad():
    tok = itransformer.embed_tokens(last_window.to(device))
    layer0 = itransformer.encoder.layers[0]
    normed = layer0.norm1(tok) if layer0.norm_first else tok
    _, attn_weights = layer0.self_attn(normed, normed, normed, need_weights=True, average_attn_weights=True)
    attn_weights = attn_weights[0].cpu().numpy()

attn_df = pd.DataFrame(attn_weights, index=feature_cols, columns=feature_cols)
attn_df.to_csv("attention_weights_hybrid_v2.csv")

fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(attn_df.values, cmap="viridis", aspect="auto")
ax.set_xticks(range(len(feature_cols))); ax.set_xticklabels(feature_cols, rotation=90, fontsize=7)
ax.set_yticks(range(len(feature_cols))); ax.set_yticklabels(feature_cols, fontsize=7)
ax.set_title("iTransformer Attention Weights (Hybrid v2, 18-parameter model)", fontsize=12)
fig.colorbar(im, ax=ax, label="attention weight")
fig.tight_layout()
fig.savefig("attention_map_hybrid_v2.png", dpi=110, bbox_inches="tight")
plt.show()


## 10. Save outputs for the dashboard

In [ ]:
fva = pd.DataFrame({"timestamp": truth.index})
for p in report_params:
    fva[f"{p}__actual"] = truth[p].values
    fva[f"{p}__hybrid_v2"] = hybrid_final[p].values
    fva[f"{p}__engine"] = ENGINE[p]
fva.to_csv("forecast_vs_actual_hybrid_v2.csv", index=False)

dup_fva = pd.DataFrame({"timestamp": test_df.index})
for keep, drop in DUPLICATES:
    dup_fva[f"{drop}__actual"] = df_10min[drop].iloc[-HORIZON:].values
    dup_fva[f"{drop}__reconstructed"] = dup_series[drop]
dup_fva.to_csv("duplicate_forecast_vs_actual_hybrid_v2.csv", index=False)

dup_recon_rows = []
for keep, drop in DUPLICATES:
    _, slope, intercept, r2 = recon_coef[drop]
    mae = mean_absolute_error(df_10min[drop].iloc[-HORIZON:].values, dup_series[drop])
    rmse = np.sqrt(mean_squared_error(df_10min[drop].iloc[-HORIZON:].values, dup_series[drop]))
    dup_recon_rows.append({"duplicate_parameter": drop, "reconstructed_from": keep,
                            "engine": ENGINE[keep], "slope": round(slope, 4),
                            "intercept": round(intercept, 4), "train_R2": round(r2, 5),
                            "held_out_MAE": round(mae, 4), "held_out_RMSE": round(rmse, 4)})
pd.DataFrame(dup_recon_rows).to_csv("duplicate_reconstruction_hybrid_v2.csv", index=False)

print("Saved: metrics_hybrid_v2.csv, forecast_vs_actual_hybrid_v2.csv, duplicate_reconstruction_hybrid_v2.csv,")
print("       duplicate_forecast_vs_actual_hybrid_v2.csv, attention_weights_hybrid_v2.csv, plot PNG.")


## 11. Conclusion

The distribution-correct loss function (Tweedie for zero-inflated precipitation,
Pseudo-Huber for ceiling-saturated visibility, both on a tree model that physically
cannot extrapolate past its training range) is the literature-grounded fix, not a
different recurrent/transformer architecture wrapped around the same wrong (Gaussian)
likelihood. Section 7's table is the actual verdict: compare `hybrid_v2_skill_%` against
both `pure_iTransformer_skill_%` and `deepar_hybrid_skill_%` directly. If `hybrid_v2`
beats `deepar_hybrid` on most/all of the 6, that confirms the diagnosis — these
parameters needed the right *distributional assumption*, not a fancier sequence model.
